In [1]:
# load the libraries
import pandas as pd
import numpy as np

## Adding npm download count for each package version to the dataset

In [2]:
# Load the data
data = pd.read_csv('../../package_data.csv')
npm_data = data[data['System']=='NPM'].copy()
full_package_name = npm_data['package_name'].unique()

## save to a text file
with open('npm_full_package_names.txt', 'w') as f:
    for name in full_package_name:
        f.write(f"{name}\n")

In [ ]:
import requests
import os
import csv
import time
from datetime import datetime
from urllib.parse import quote_plus

# --- API Endpoints ---
VERSION_DOWNLOADS_API = "https://api.npmjs.org/versions/{package}/last-week"
REGISTRY_METADATA_API = "https://registry.npmjs.org/{package}"
NPM_TOKEN = 'xxxx'

# --- Configuration ---
PACKAGES_FILE_PATH = 'npm_full_package_names.txt' 
CSV_OUTPUT_FILE = 'npm_version_downloads.csv'
FAILED_PACKAGES_FILE = 'npm_failed_packages.csv'
PACKAGES_PER_CHECKPOINT = 50 
API_REQUEST_DELAY = 0.5 # seconds


# --- File Reading Function (No Change) ---

def read_packages_from_file(file_path):
    if not os.path.exists(file_path):
        print(f"\nERROR: Package list file not found at: {file_path}")
        return []
        
    print(f"\nReading packages from file: {file_path}...")
    packages = []
    
    try:
        with open(file_path, 'r') as f:
            for line in f:
                package_name = line.strip()
                if package_name:
                    packages.append(package_name)
        return packages
    except IOError as e:
        print(f"Error reading file {file_path}: {e}")
        return []

# --- API Functions ------------------------------------------------------

def get_all_versions_7day_downloads(package_name):
    encoded_package_name = quote_plus(package_name)
    api_url = VERSION_DOWNLOADS_API.format(package=encoded_package_name)

    try:
        headers = {
            'Authorization': f'Bearer {NPM_TOKEN}'
        }
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get('downloads', {})
    except requests.exceptions.HTTPError as e:
        # Specifically catch 404 (Not Found) or 400 (Bad Request)
        if response.status_code in [404, 400]:
            return {'error': 'Package Not Found/Bad Request'}
        else:
            return {'error': f'HTTP Error {response.status_code}'}
    except requests.exceptions.RequestException as e:
        return {'error': f'Network Error: {e.__class__.__name__}'}


def get_package_metadata_versions(package_name):
    encoded_package_name = quote_plus(package_name)
    api_url = REGISTRY_METADATA_API.format(package=encoded_package_name)
    try:
        headers = {'Accept': 'application/vnd.npm.install-v1+json; q=1.0, application/json; q=0.9, */*; q=0.8',
                   'Authorization': f'Bearer {NPM_TOKEN}'}
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        if 'versions' in data:
            return list(data['versions'].keys())
        return []
    except requests.exceptions.RequestException:
        # This function is secondary, returning empty list is fine for errors here
        return []

# --- Data Processing Function (Modified to return status and handle errors) ---

def fetch_and_process_version_downloads(package_name):
    """
    Fetches download data for a package.
    Returns: ('success', list_of_data) or ('fail', error_reason).
    """
    time.sleep(API_REQUEST_DELAY) 
    
    downloads_per_version = get_all_versions_7day_downloads(package_name)

    # Check for error returned by the API call
    if isinstance(downloads_per_version, dict) and 'error' in downloads_per_version:
        return ('fail', downloads_per_version['error'])

    all_versions = get_package_metadata_versions(package_name)
    if not all_versions:
        # This could happen if package is private or very old and metadata fetch fails
        return ('fail', 'Could not retrieve any versions from metadata API.')
        
    current_timestamp = datetime.now().isoformat()
    report_data = []

    for version in all_versions:
        downloads = downloads_per_version.get(version, 0)
        
        report_data.append({
            'package_name': package_name,
            'version': version,
            'downloads_7_day_total': downloads,
            'retrieval_date': current_timestamp
        })
            
    return ('success', report_data)


# --- Checkpoint/CSV Writing Function (Unchanged, writes successful data) ---

def write_data_to_csv(data_list, filename):
    # ... (function is identical to previous version, handles appending success data)
    if not data_list:
        return

    file_exists = os.path.exists(filename)
    fieldnames = list(data_list[0].keys())

    try:
        with open(filename, 'a', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            if not file_exists:
                writer.writeheader()
                
            writer.writerows(data_list)
            
        print(f"   [SAVED] Checkpoint complete. Wrote {len(data_list)} rows.")
        
    except Exception as e:
        print(f"\n❌ ERROR writing to CSV file '{filename}': {e}")


# --- Failure Logging Function ----------------------------------------------

def write_failed_package_to_csv(package_name, error_reason, filename):
    """Writes a single failed package record to the dedicated error log file."""
    
    data = [{
        'package_name': package_name,
        'error_reason': error_reason,
        'retrieval_date': datetime.now().isoformat()
    }]
    
    file_exists = os.path.exists(filename)
    fieldnames = list(data[0].keys())

    try:
        # Use 'a' mode for appending
        with open(filename, 'a', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            # Write header only if the file is new
            if not file_exists:
                writer.writeheader()
                
            writer.writerows(data)
            
        print(f"   [FAILED] Logged error: {error_reason}")
        
    except Exception as e:
        # This is a critical failure, we just print and move on
        print(f"\n❌ CRITICAL ERROR writing failure log: {e}")


# --- Execution -------------------------------------------------

if __name__ == "__main__":
    
    package_names = read_packages_from_file(PACKAGES_FILE_PATH)
    
    if not package_names:
        print("\nExiting script.")
    else:
        print(f"\nSuccessfully loaded {len(package_names)} packages.")
        print(f"Checkpoints will be saved every {PACKAGES_PER_CHECKPOINT} packages.")
        print("-" * 60)
        
        all_package_data_buffer = []
        total_rows_written = 0
        total_failures = 0
        
        for i, pkg in enumerate(package_names):
            current_package_index = i + 1
            print(f"--- [ {current_package_index}/{len(package_names)} ] Analyzing: {pkg} ---")
            
            # Fetch data for the current package
            status, result = fetch_and_process_version_downloads(pkg)
            
            if status == 'success':
                # Add results to the success buffer
                all_package_data_buffer.extend(result)
            else:
                # Log the failure immediately
                total_failures += 1
                write_failed_package_to_csv(pkg, result, FAILED_PACKAGES_FILE)


            # --- Checkpoint Logic ---
            if current_package_index % PACKAGES_PER_CHECKPOINT == 0 or current_package_index == len(package_names):
                
                if all_package_data_buffer:
                    # Write the contents of the buffer to the success CSV
                    write_data_to_csv(all_package_data_buffer, CSV_OUTPUT_FILE)
                    total_rows_written += len(all_package_data_buffer)
                    all_package_data_buffer = [] # Clear the buffer
                else:
                    print("   [SKIP] Buffer empty, skipping checkpoint write.")
        
        print("\n" + "="*60)
        print("✅ SCRIPT COMPLETE")
        print(f"Total packages processed: {len(package_names)}")
        print(f"Total rows (versions) written to '{CSV_OUTPUT_FILE}': {total_rows_written:,}")
        print(f"Total package failures logged to '{FAILED_PACKAGES_FILE}': {total_failures}")
        print("="*60)